# AlphaZero Connect Four — Colab 訓練
薄封裝：所有邏輯都在 `az` 套件裡，這本 notebook 只負責環境、背景執行、監控與收尾。

**使用方式**
1. Runtime → Change runtime type → **A100 GPU**。
2. 左側 🔑 Secrets 加入 `HF_TOKEN`（write 權限；沒有也能訓練，只是結束時不自動 push）。
3. 下方參數 cell 先跑一次 `SMOKE_TEST = True`（約 10 分鐘，驗證管線 + 實測 A100 吞吐），再改 `False` Run all 過夜。

**斷線防護**：checkpoint 每個 iteration 原子寫入 Drive；斷線後重開 notebook、Run all 即自動 `--resume` 續跑。

In [ ]:
#@title 參數
SMOKE_TEST = True   #@param {type:"boolean"}
HF_USERNAME = "steven0226"
RUN_NAME = "run1"
MAX_HOURS = 8.0     # 正式訓練的 wallclock 預算（優雅收尾，把 push 留在額度內）

# 原始碼來源：GitHub repo；若尚未發佈，把整個專案資料夾放到
# Drive 的 MyDrive/alphazero-connect4-src，會自動 fallback
REPO_URL = "https://github.com/steven0226/alphazero-connect4.git"


In [ ]:
#@title GPU / Drive / 原始碼 / 安裝
import os, shutil, subprocess, zipfile

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.isdir("alphazero-connect4"):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "alphazero-connect4"])
    if r.returncode != 0:
        # GitHub repo not published yet: fall back to a source-only zip on
        # Drive (upload alphazero-connect4-src.zip to MyDrive/, generated
        # locally by the project so it excludes .venv/checkpoints/.git)
        zip_path = "/content/drive/MyDrive/alphazero-connect4-src.zip"
        assert os.path.isfile(zip_path), (
            f"git clone 失敗且 {zip_path} 不存在——"
            "先把 alphazero-connect4-src.zip 上傳到 Google Drive 的 MyDrive 根目錄"
        )
        os.makedirs("alphazero-connect4", exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("alphazero-connect4")
        print("source: Drive zip fallback")
%cd /content/alphazero-connect4
# torch 用 Colab 內建 CUDA 版（pyproject 只設下界，不會被覆蓋）
!pip install -q -e . pytest safetensors


In [ ]:
#@title HF token（從 Colab Secrets 讀，永不寫進 notebook）+ 煙霧測試
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded ✓")
except Exception:
    print("⚠ 沒有 HF_TOKEN secret：訓練照跑，結束時自動 push 會跳過（權重仍在 Drive）")

# 同一套單元測試在 Colab 再跑一次，抓環境漂移
!python -m pytest tests -q


## A100 過夜預算估算（正式設定的依據）

每 iteration 的葉評估數 ≈ `128 局 × ~25 手/局 × 160 sims ≈ 512k`。
A100 上 batch-128 的單次前向（6 blocks × 96 filters、6×7 輸入）約 1–3 ms，
但瓶頸在 Python 樹操作——實測有效吞吐約 **15–30k evals/s**
（本機 RTX 2070 已實測 23k evals/s 純推理；A100 推理更快、樹操作同速）。

- 自我對弈 ≈ 512k ÷ 20k ≈ **25–50 s/iter**
- arena（3 錨點 × 40 局，批次推理 + CPU rollout 錨點）≈ **30–60 s/iter**
- 訓練 64 step × batch 256 ≈ **<10 s**

→ **約 1.5–3 分鐘/iteration** → 6–10 小時 ≈ **150–280 iterations**。
`FULL` preset 目標 **150 iterations**（`--max-hours 8` 兜底：時間到就存檔收尾）。
`SMOKE_TEST` 模式跑 3 個迷你 iteration 就是你的 A100 實測——用實際 sec/iter 外推再開正式 run。

In [ ]:
#@title 啟動訓練（背景 subprocess，log 落 Drive）
import subprocess, sys
from pathlib import Path

CKPT_DIR = Path(f"/content/drive/MyDrive/alphazero-connect4/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else ""))
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    args = ["--preset", "smoke", "--iterations", "3", "--games-per-iter", "8"]
else:
    args = ["--preset", "full", "--max-hours", str(MAX_HOURS)]

log_path = CKPT_DIR / "train.log"
log_file = open(log_path, "a")
proc = subprocess.Popen(
    [sys.executable, "-m", "az.train", *args, "--ckpt-dir", str(CKPT_DIR), "--resume"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"training started: pid={proc.pid}  preset={'smoke' if SMOKE_TEST else 'full'}")
print(f"log: {log_path}")


In [ ]:
#@title 監控（可隨時中斷/重跑此 cell，不影響訓練）
import time
from pathlib import Path
from IPython.display import clear_output, Image as IPImage, display

while proc.poll() is None:
    clear_output(wait=True)
    log = Path(CKPT_DIR / "train.log")
    if log.exists():
        print(log.read_text()[-2500:])
    curve = CKPT_DIR / "elo_curve.png"
    if curve.exists():
        display(IPImage(str(curve)))
    time.sleep(60)
print(f"training process exited with code {proc.returncode}")
print(Path(CKPT_DIR / "train.log").read_text()[-1500:])


In [ ]:
#@title 收尾：GIF + push 最佳權重/Elo 曲線/model card 到 HF，然後釋放 runtime
import os, traceback

try:
    proc.wait()
    if os.environ.get("HF_TOKEN") and not SMOKE_TEST:
        # 自我對弈一局渲染成 GIF，進 model card
        !python scripts/make_gif.py --ckpt "{CKPT_DIR}/best.pt" --out assets/demo.gif
        !python scripts/push_model.py --ckpt-dir "{CKPT_DIR}" --repo-id "{HF_USERNAME}/alphazero-connect4"
    else:
        print(f"skip push（SMOKE_TEST 或無 token）。權重與 elo.csv 都在 {CKPT_DIR}")
except Exception:
    traceback.print_exc()
    print(f"push 失敗不影響資料：Drive 是真相來源 -> {CKPT_DIR}")
finally:
    from google.colab import runtime
    runtime.unassign()   # 一定釋放，不燒額度
